# 03e — Baseline Logistic Regression (Fase 3: Modelado)

Quinto notebook de modelado del TFG. Entrena **8 modelos** (4 versiones de
master × 2 targets) con LogReg con regularización L2 (Ridge) y defaults razonables, sin tuning de
hiperparámetros. Es un baseline de comparación.

## Preguntas que responde
1. ¿Qué versión de master funciona mejor?
2. ¿Qué features están saliendo como más importantes?
3. ¿Cuál es el orden de magnitud del rendimiento alcanzable?

## Setup
- **8 modelos**: 4 versiones × 2 targets
- **Splits**: 70/15/15 estratificados por `churn_30d`, `random_state=42`,
  compartidos entre los 8 modelos para garantizar comparabilidad
- **Categóricas**: dtype `category` (LogReg con regularización L2 (Ridge) las maneja nativamente)
- **NaN preservados**: LogReg con regularización L2 (Ridge) los maneja
- **Threshold**: 0.5 (sin búsqueda de óptimo en este baseline)
- **Early stopping**: 50 rondas sin mejora en AUC sobre validation

## Outputs
- `data/data_models/splits_indices.parquet` (compartido)
- `data/data_models/model_<version>_<target>.txt` (8 modelos)
- `data/data_models/predictions_<version>_<target>.parquet` (8 archivos)
- `informes/fase3_modeling/03a_logreg/baseline_comparison/` (métricas, importance, gráficos, reports)


In [1]:
# [SETUP] Imports
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss, f1_score,
    confusion_matrix, roc_curve, precision_recall_curve,
)
import sklearn
from pathlib import Path
import json
import gc
import time
from datetime import datetime

assert sklearn.__version__ >= '1.0', f'sklearn >= 1.0 requerido, instalado {sklearn.__version__}'
print(f'sklearn {sklearn.__version__} OK')


sklearn 1.8.0 OK


In [2]:
# [SETUP] Paths y constantes
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase3_modeling' else Path.cwd()
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
DATA_MODELS = PROJECT_ROOT / 'data' / 'data_models'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase3_modeling' / '03e_logreg' / 'baseline_comparison'
DATA_MODELS.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15
TRAIN_SIZE = 0.70
REFERENCE_DATE = pd.Timestamp('2026-04-04', tz='UTC')
CUTOFF_DATE = REFERENCE_DATE - pd.Timedelta(days=120)
# N_SAMPLE se lee dinámicamente del primer parquet (cell 4)

TARGETS = ['churn_14d', 'churn_30d']
MASTER_VERSIONS = {
    'original':         DATA_QC / 'master_table_cutoff.parquet',
    'v1_conservative':  DATA_QC / 'master_table_cutoff_v1_conservative.parquet',
    'v2_intermediate':  DATA_QC / 'master_table_cutoff_v2_intermediate.parquet',
    'v3_aggressive':    DATA_QC / 'master_table_cutoff_v3_aggressive.parquet',
}

# Validar inputs
for name, path in MASTER_VERSIONS.items():
    assert path.exists(), f'STOP: master {name} no encontrada en {path}'

steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f'[{tag}] {ts} — {message}'
    steps_log.append(entry)
    print(entry)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_MODELS:  {DATA_MODELS}')
print(f'REPORT_DIR:   {REPORT_DIR}')
print(f'\nMaster versions: {len(MASTER_VERSIONS)}')
for n, p in MASTER_VERSIONS.items():
    print(f'  {n:<18}: {p.stat().st_size/1e6:.1f} MB')


PROJECT_ROOT: /Users/jezquerro/Documents/tfg
DATA_MODELS:  /Users/jezquerro/Documents/tfg/data/data_models
REPORT_DIR:   /Users/jezquerro/Documents/tfg/informes/fase3_modeling/03e_logreg/baseline_comparison

Master versions: 4
  original          : 4.1 MB
  v1_conservative   : 2.7 MB
  v2_intermediate   : 2.7 MB
  v3_aggressive     : 2.3 MB


## BLOQUE 2 — Splits unificados

In [3]:
# [EXEC] 2.1-2.2 Cargar master original y definir splits estratificados
master = pd.read_parquet(MASTER_VERSIONS['original'])
N_SAMPLE = len(master)  # dinámico
assert N_SAMPLE > 20_000, f'sample sospechosamente pequeño: {N_SAMPLE}'
assert master['user_id'].is_unique

# Asegurar índice canónico [0..N-1] para que los splits indexen sin sorpresas
master = master.reset_index(drop=True)

# Targets como int (eran bool en la master post-02zz; LogReg acepta ambos)
y_30d = master['churn_30d'].astype(int)
y_14d = master['churn_14d'].astype(int)
user_ids = master['user_id'].copy()

# Split 1: separar test (15%) del resto (85%)
idx_all = np.arange(N_SAMPLE)
idx_trainval, idx_test = train_test_split(
    idx_all,
    test_size=TEST_SIZE,
    stratify=y_30d,
    random_state=RANDOM_STATE,
)

# Split 2: del 85% restante, sacar val (15% del total → ~17.65% del trainval)
val_size_relative = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
idx_train, idx_val = train_test_split(
    idx_trainval,
    test_size=val_size_relative,
    stratify=y_30d.iloc[idx_trainval],
    random_state=RANDOM_STATE,
)

# Validar tamaños
assert abs(len(idx_train) / N_SAMPLE - TRAIN_SIZE) < 0.01
assert abs(len(idx_val) / N_SAMPLE - VAL_SIZE) < 0.01
assert abs(len(idx_test) / N_SAMPLE - TEST_SIZE) < 0.01

# Validar estratificación
churn_train = float(y_30d.iloc[idx_train].mean())
churn_val = float(y_30d.iloc[idx_val].mean())
churn_test = float(y_30d.iloc[idx_test].mean())
assert abs(churn_train - churn_val) < 0.01, f'estratificación falla: train={churn_train}, val={churn_val}'
assert abs(churn_train - churn_test) < 0.01, f'estratificación falla: train={churn_train}, test={churn_test}'

print(f'Splits:')
print(f'  train: {len(idx_train):,} ({len(idx_train)/N_SAMPLE*100:.1f}%)')
print(f'  val:   {len(idx_val):,}  ({len(idx_val)/N_SAMPLE*100:.1f}%)')
print(f'  test:  {len(idx_test):,}  ({len(idx_test)/N_SAMPLE*100:.1f}%)')
print(f'\nChurn 30d en cada split:')
print(f'  train: {churn_train:.4f}')
print(f'  val:   {churn_val:.4f}')
print(f'  test:  {churn_test:.4f}')

log_step('EXEC', f'Splits: train={len(idx_train)}, val={len(idx_val)}, test={len(idx_test)}')
log_step('ANALYSIS', f'Churn 30d: train={churn_train:.4f}, val={churn_val:.4f}, test={churn_test:.4f}')


Splits:
  train: 17,639 (70.0%)
  val:   3,781  (15.0%)
  test:  3,780  (15.0%)

Churn 30d en cada split:
  train: 0.4742
  val:   0.4742
  test:  0.4743
[EXEC] 19:36:45 — Splits: train=17639, val=3781, test=3780
[ANALYSIS] 19:36:45 — Churn 30d: train=0.4742, val=0.4742, test=0.4743


In [4]:
# [EXEC] 2.3 Guardar splits
splits_df = pd.DataFrame({
    'user_id': user_ids,
    'split': pd.Series(['train'] * N_SAMPLE),
})
splits_df.loc[idx_val, 'split'] = 'val'
splits_df.loc[idx_test, 'split'] = 'test'

assert splits_df['split'].value_counts().sum() == N_SAMPLE
print(f'Distribución splits: {splits_df["split"].value_counts().to_dict()}')

splits_path = DATA_MODELS / 'splits_indices_cutoff.parquet'
splits_df.to_parquet(splits_path, engine='pyarrow', index=False)
print(f'Splits guardados: {splits_path}')
log_step('EXEC', f'splits_indices_cutoff.parquet guardado en {splits_path}')

# Liberar master original (cada modelo recarga su versión)
del master
gc.collect()


Distribución splits: {'train': 17639, 'val': 3781, 'test': 3780}
Splits guardados: /Users/jezquerro/Documents/tfg/data/data_models/splits_indices_cutoff.parquet
[EXEC] 19:36:45 — splits_indices_cutoff.parquet guardado en /Users/jezquerro/Documents/tfg/data/data_models/splits_indices_cutoff.parquet


24

## BLOQUE 3 — Función de entrenamiento reutilizable

In [5]:
# [EXEC] Función de entrenamiento (Logistic Regression con preprocesamiento)
def train_logreg_baseline(master_path, target_col, version_name, target_name, splits_df):
    """LR con: imputación (median num, 'missing' cat) + StandardScaler + OneHotEncoder + L2.

    Convención: feature importance = abs(coef) sobre features post-preprocesamiento.
    """
    t0 = time.time()

    df = pd.read_parquet(master_path).reset_index(drop=True)
    assert len(df) == N_SAMPLE

    target_cols_to_drop = ['churn_14d', 'churn_30d', 'user_id']
    y = df[target_col].astype(int)
    X = df.drop(columns=target_cols_to_drop)

    # Detectar tipos
    cat_cols = X.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    num_cols = X.select_dtypes(include=['int64', 'int32', 'float64', 'float32', 'Int64', 'Int32', 'bool']).columns.tolist()
    # bool → cast a int (algunos imputers fallan con bool nullable)
    for c in num_cols:
        if X[c].dtype == 'bool':
            X[c] = X[c].astype('int8')

    train_mask = splits_df['split'].values == 'train'
    val_mask = splits_df['split'].values == 'val'
    test_mask = splits_df['split'].values == 'test'

    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val = X[val_mask], y[val_mask]
    X_test, y_test = X[test_mask], y[test_mask]

    preprocessor = ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), cat_cols),
    ], remainder='drop', verbose_feature_names_out=False)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            C=1.0, max_iter=2000, random_state=RANDOM_STATE,
            solver='liblinear', penalty='l2',
        )),
    ])

    pipeline.fit(X_train, y_train)

    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    auc_roc = float(roc_auc_score(y_test, y_pred_proba))
    auc_pr = float(average_precision_score(y_test, y_pred_proba))
    ll = float(log_loss(y_test, y_pred_proba))
    f1 = float(f1_score(y_test, y_pred))

    n_top = max(1, int(0.10 * len(y_test)))
    top_idx = np.argsort(y_pred_proba)[-n_top:]
    recall_at_10 = float(y_test.iloc[top_idx].sum()) / max(1, int(y_test.sum()))

    cm = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    precision_c, recall_c, _ = precision_recall_curve(y_test, y_pred_proba)

    # Feature importance: abs(coef_) sobre features expandidas. Agregamos a nivel raw feature
    # mediante mapeo desde get_feature_names_out al feature original.
    expanded_names = preprocessor.get_feature_names_out()
    coefs = np.abs(pipeline.named_steps['classifier'].coef_[0])
    # Mapear cada feature expandida a su raw feature
    raw_for_expanded = []
    for n in expanded_names:
        # numéricas: nombre directo. Categóricas: 'col_value'
        if n in num_cols:
            raw_for_expanded.append(n)
        else:
            # Buscar prefijo entre cat_cols
            matched = None
            for c in cat_cols:
                if n.startswith(c + '_') or n == c:
                    matched = c
                    break
            raw_for_expanded.append(matched if matched else n)
    fi_raw = pd.Series(coefs, index=raw_for_expanded).groupby(level=0).sum()
    fi_df = pd.DataFrame({
        'feature': fi_raw.index,
        'importance_gain': fi_raw.values,  # |coef| sumado por feature raw
        'importance_split': [1] * len(fi_raw),  # placeholder
    }).sort_values('importance_gain', ascending=False).reset_index(drop=True)

    elapsed = time.time() - t0
    log_step('EXEC', f'[{version_name}/{target_name}] AUC={auc_roc:.4f} | n_features_expanded={len(expanded_names)} | t={elapsed:.1f}s')

    return {
        'version': version_name,
        'target': target_name,
        'n_features': X.shape[1],
        'n_categorical': len(cat_cols),
        'best_iteration': pipeline.named_steps['classifier'].n_iter_[0] if hasattr(pipeline.named_steps['classifier'], 'n_iter_') else 0,
        'auc_roc': auc_roc,
        'auc_pr': auc_pr,
        'log_loss': ll,
        'f1': f1,
        'recall_at_10': recall_at_10,
        'confusion_matrix': cm,
        'fpr': fpr,
        'tpr': tpr,
        'precision_curve': precision_c,
        'recall_curve': recall_c,
        'feature_importance': fi_df,
        'model': pipeline,
        'predictions': pd.DataFrame({
            'user_id': df.loc[test_mask, 'user_id'].values,
            'y_true': y_test.values,
            'y_pred_proba': y_pred_proba,
            'y_pred': y_pred,
        }),
        'elapsed_seconds': elapsed,
    }

print('Función train_logreg_baseline definida')


Función train_logreg_baseline definida


## BLOQUE 4 — Entrenar los 8 modelos

In [6]:
# [EXEC] Entrenar 4 versiones × 2 targets = 8 modelos
splits_df = pd.read_parquet(DATA_MODELS / 'splits_indices_cutoff.parquet')

results = {}
total_start = time.time()
for version_name, master_path in MASTER_VERSIONS.items():
    for target_col in TARGETS:
        key = f'{version_name}__{target_col}'
        log_step('EXEC', f'>>> Entrenando {key}')
        results[key] = train_logreg_baseline(
            master_path=master_path,
            target_col=target_col,
            version_name=version_name,
            target_name=target_col,
            splits_df=splits_df,
        )
        gc.collect()

total_elapsed = time.time() - total_start
print(f'\n=== 8 modelos entrenados en {total_elapsed:.1f}s ({total_elapsed/60:.1f} min) ===')
log_step('EXEC', f'8 modelos entrenados en {total_elapsed:.1f}s')


[EXEC] 19:36:45 — >>> Entrenando original__churn_14d


/Users/jezquerro/Documents/tfg/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[EXEC] 19:36:49 — [original/churn_14d] AUC=0.9993 | n_features_expanded=53227 | t=3.3s
[EXEC] 19:36:49 — >>> Entrenando original__churn_30d


/Users/jezquerro/Documents/tfg/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[EXEC] 19:36:52 — [original/churn_30d] AUC=0.9999 | n_features_expanded=53227 | t=3.8s
[EXEC] 19:36:52 — >>> Entrenando v1_conservative__churn_14d


...

[EXEC] 19:37:07 — [v2_intermediate/churn_30d] AUC=0.7544 | n_features_expanded=17926 | t=3.8s
[EXEC] 19:37:07 — >>> Entrenando v3_aggressive__churn_14d


/Users/jezquerro/Documents/tfg/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[EXEC] 19:37:09 — [v3_aggressive/churn_14d] AUC=0.7891 | n_features_expanded=17915 | t=2.3s
[EXEC] 19:37:09 — >>> Entrenando v3_aggressive__churn_30d


/Users/jezquerro/Documents/tfg/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[EXEC] 19:37:12 — [v3_aggressive/churn_30d] AUC=0.7479 | n_features_expanded=17915 | t=2.6s

=== 8 modelos entrenados en 26.5s (0.4 min) ===
[EXEC] 19:37:12 — 8 modelos entrenados en 26.5s


## BLOQUE 5 — Guardado de modelos, predicciones y feature importance

In [7]:
# [EXEC] Guardado
saved = []
for key, res in results.items():
    version, target = key.split('__')

    # Modelo nativo LogReg
    model_path = DATA_MODELS / f'model_{version}_{target}_cutoff.txt'
    import joblib; joblib.dump(res['model'], str(model_path))

    # Predicciones
    pred_path = DATA_MODELS / f'predictions_{version}_{target}_cutoff.parquet'
    res['predictions'].to_parquet(pred_path, engine='pyarrow', index=False)

    # Feature importance
    fi_path = REPORT_DIR / f'feature_importance_{version}_{target}.csv'
    res['feature_importance'].to_csv(fi_path, index=False)

    saved.append((key, model_path.stat().st_size/1024, pred_path.stat().st_size/1024))

print('Modelos y predicciones guardados:')
for k, m_kb, p_kb in saved:
    print(f'  {k:<40s}  model={m_kb:>6.0f} KB  preds={p_kb:>6.0f} KB')
log_step('EXEC', f'8 modelos + predicciones + feature_importance guardados')


Modelos y predicciones guardados:
  original__churn_14d                       model=  1830 KB  preds=   126 KB
  original__churn_30d                       model=  1830 KB  preds=   126 KB
  v1_conservative__churn_14d                model=   621 KB  preds=   126 KB
  v1_conservative__churn_30d                model=   621 KB  preds=   126 KB
  v2_intermediate__churn_14d                model=   621 KB  preds=   126 KB
  v2_intermediate__churn_30d                model=   621 KB  preds=   126 KB
  v3_aggressive__churn_14d                  model=   620 KB  preds=   126 KB
  v3_aggressive__churn_30d                  model=   620 KB  preds=   126 KB
[EXEC] 19:37:12 — 8 modelos + predicciones + feature_importance guardados


## BLOQUE 6 — Tabla comparativa de métricas

In [8]:
# [ANALYSIS] Tabla comparativa
rows = []
for key, res in results.items():
    rows.append({
        'version': res['version'],
        'target': res['target'],
        'n_features': res['n_features'],
        'best_iteration': res['best_iteration'],
        'auc_roc': round(res['auc_roc'], 4),
        'auc_pr': round(res['auc_pr'], 4),
        'log_loss': round(res['log_loss'], 4),
        'f1': round(res['f1'], 4),
        'recall_at_10': round(res['recall_at_10'], 4),
        'tn': int(res['confusion_matrix'][0, 0]),
        'fp': int(res['confusion_matrix'][0, 1]),
        'fn': int(res['confusion_matrix'][1, 0]),
        'tp': int(res['confusion_matrix'][1, 1]),
        'elapsed_s': round(res['elapsed_seconds'], 1),
    })
metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(REPORT_DIR / 'metrics_comparison.csv', index=False)

print('=== metrics_comparison.csv guardado ===')
for target in TARGETS:
    print(f'\n--- Resultados para {target} (orden desc por AUC-ROC) ---')
    sub = metrics_df[metrics_df['target'] == target].sort_values('auc_roc', ascending=False)
    print(sub.to_string(index=False))

log_step('ANALYSIS', f'metrics_comparison.csv: {len(metrics_df)} filas')


=== metrics_comparison.csv guardado ===

--- Resultados para churn_14d (orden desc por AUC-ROC) ---
        version    target  n_features  best_iteration  auc_roc  auc_pr  log_loss     f1  recall_at_10   tn  fp  fn   tp  elapsed_s
       original churn_14d         112              31   0.9993  0.9941    0.0440 0.9890        0.2966 2490  19   9 1262        3.3
v1_conservative churn_14d          93              64   0.7966  0.7050    0.5104 0.6521        0.2565 2149 360 482  789        3.3
v2_intermediate churn_14d          88              77   0.7964  0.7078    0.5106 0.6515        0.2557 2151 358 484  787        2.9
  v3_aggressive churn_14d          77              80   0.7891  0.6846    0.5196 0.6347        0.2463 2130 379 504  767        2.3

--- Resultados para churn_30d (orden desc por AUC-ROC) ---
        version    target  n_features  best_iteration  auc_roc  auc_pr  log_loss     f1  recall_at_10   tn  fp  fn   tp  elapsed_s
       original churn_30d         112              12 

## BLOQUE 7 — Visualizaciones

In [9]:
# [ANALYSIS] 7.1 Curvas ROC
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
COLORS = {'original': '#2c3e50', 'v1_conservative': '#3498db',
          'v2_intermediate': '#e67e22', 'v3_aggressive': '#c0392b'}

for ax_idx, target in enumerate(TARGETS):
    ax = axes[ax_idx]
    for version_name in MASTER_VERSIONS.keys():
        key = f'{version_name}__{target}'
        res = results[key]
        ax.plot(res['fpr'], res['tpr'],
                label=f"{version_name} (AUC={res['auc_roc']:.3f})",
                linewidth=2, color=COLORS[version_name])
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves — {target}')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'roc_curves.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'roc_curves.png guardado')
log_step('ANALYSIS', '7.1 roc_curves.png')


roc_curves.png guardado
[ANALYSIS] 19:37:12 — 7.1 roc_curves.png


In [10]:
# [ANALYSIS] 7.2 Curvas PR
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, target in enumerate(TARGETS):
    ax = axes[ax_idx]
    for version_name in MASTER_VERSIONS.keys():
        key = f'{version_name}__{target}'
        res = results[key]
        ax.plot(res['recall_curve'], res['precision_curve'],
                label=f"{version_name} (AP={res['auc_pr']:.3f})",
                linewidth=2, color=COLORS[version_name])
    baseline = float(results[f'original__{target}']['predictions']['y_true'].mean())
    ax.axhline(baseline, color='k', linestyle='--', alpha=0.3, label=f'Baseline ({baseline:.2f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'PR Curves — {target}')
    ax.legend(loc='lower left', fontsize=9)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'pr_curves.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'pr_curves.png guardado')
log_step('ANALYSIS', '7.2 pr_curves.png')


pr_curves.png guardado
[ANALYSIS] 19:37:12 — 7.2 pr_curves.png


In [11]:
# [ANALYSIS] 7.3 Confusion matrices (8 subplots)
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for col_idx, version_name in enumerate(MASTER_VERSIONS.keys()):
    for row_idx, target in enumerate(TARGETS):
        key = f'{version_name}__{target}'
        cm = results[key]['confusion_matrix']
        ax = axes[row_idx, col_idx]
        im = ax.imshow(cm, cmap='Blues')
        ax.set_title(f'{version_name}\n{target} (AUC={results[key]["auc_roc"]:.3f})', fontsize=10)
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(['No churn', 'Churn'])
        ax.set_yticklabels(['No churn', 'Churn'])
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True' if col_idx == 0 else '')
        max_val = cm.max()
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                        color='white' if cm[i, j] > max_val/2 else 'black',
                        fontsize=10)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'confusion_matrices.png guardado')
log_step('ANALYSIS', '7.3 confusion_matrices.png')


confusion_matrices.png guardado
[ANALYSIS] 19:37:12 — 7.3 confusion_matrices.png


In [12]:
# [ANALYSIS] 7.4 Heatmap top 30 features × 8 modelos
top_n = 30
all_top_features = set()
for key, res in results.items():
    top_feats = res['feature_importance'].head(top_n)['feature'].tolist()
    all_top_features.update(top_feats)

feature_list = sorted(all_top_features)
matrix = pd.DataFrame(index=feature_list, columns=list(results.keys()), dtype=float)

for key, res in results.items():
    fi = res['feature_importance'].copy()
    fi['rank'] = fi['importance_gain'].rank(ascending=False, method='min')
    fi_dict = dict(zip(fi['feature'], fi['rank']))
    for f in feature_list:
        matrix.loc[f, key] = fi_dict.get(f, np.nan)

# Plot
fig, ax = plt.subplots(figsize=(12, max(8, len(feature_list) * 0.25)))
masked = np.ma.masked_invalid(matrix.values.astype(float))
im = ax.imshow(masked, cmap='RdYlGn_r', aspect='auto')
ax.set_xticks(range(len(matrix.columns)))
ax.set_xticklabels([c.replace('__', '\n') for c in matrix.columns], rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(feature_list)))
ax.set_yticklabels(feature_list, fontsize=8)
plt.colorbar(im, label='Rank (1=más importante)', ax=ax)
ax.set_title(f'Top {top_n} features × 8 modelos (rank por importance_gain)')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'top_features_heatmap.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'top_features_heatmap.png guardado ({len(feature_list)} features distintas en el top {top_n} de algún modelo)')
log_step('ANALYSIS', f'7.4 top_features_heatmap.png ({len(feature_list)} features)')


top_features_heatmap.png guardado (69 features distintas en el top 30 de algún modelo)
[ANALYSIS] 19:37:13 — 7.4 top_features_heatmap.png (69 features)


In [13]:
# [ANALYSIS] Top 10 features que aparecen consistentemente en el top de los 8 modelos
top_n = 30
consensus = {}  # feature → cuántos modelos la tienen en su top_n
for key, res in results.items():
    top_feats = res['feature_importance'].head(top_n)['feature'].tolist()
    for f in top_feats:
        consensus[f] = consensus.get(f, 0) + 1

# Filtrar las que aparecen en al menos 6 de 8 modelos
robust = sorted([(f, n) for f, n in consensus.items() if n >= 6],
                key=lambda x: -x[1])

print(f'=== Features robustas (top {top_n} en >=6 de 8 modelos) ===')
print(f'  Total: {len(robust)}')
for f, n in robust[:20]:
    # Calcular rank medio entre los modelos donde aparece
    ranks = []
    for key, res in results.items():
        fi = res['feature_importance']
        if f in fi['feature'].values:
            r = fi.index[fi['feature'] == f][0] + 1
            if r <= top_n:
                ranks.append(r)
    avg_rank = np.mean(ranks) if ranks else None
    print(f'  {f:<45s} en {n}/8 modelos, rank medio={avg_rank:.1f}')

# Guardar para el report
robust_df = pd.DataFrame([{
    'feature': f,
    'n_models_top30': n,
} for f, n in robust])
robust_df.to_csv(REPORT_DIR / 'robust_top_features.csv', index=False)
log_step('ANALYSIS', f'robust_top_features.csv: {len(robust)} features')


=== Features robustas (top 30 en >=6 de 8 modelos) ===
  Total: 18
  user_created_at                               en 8/8 modelos, rank medio=1.4
  country                                       en 8/8 modelos, rank medio=2.5
  user_store_where_published                    en 8/8 modelos, rank medio=7.2
  device_primary_platform                       en 8/8 modelos, rank medio=6.9
  device_has_android                            en 8/8 modelos, rank medio=12.0
  coll_items_last_90d                           en 8/8 modelos, rank medio=20.4
  device_has_ios                                en 7/8 modelos, rank medio=10.6
  coll_total_items                              en 7/8 modelos, rank medio=16.7
  reward_current_day_max                        en 7/8 modelos, rank medio=12.1
  coll_items_last_30d                           en 7/8 modelos, rank medio=17.7
  device_unique_models                          en 7/8 modelos, rank medio=20.9
  device_days_since_last_active                 en 7/8 mo

## BLOQUE 8 — Informes

In [14]:
# [REPORT] 8.1 baseline_summary.md (resumen ejecutivo)

# Helpers para construir tablas
def metric_row(version, target, metrics_df, color_winner_col=None):
    sub = metrics_df[(metrics_df['version'] == version) & (metrics_df['target'] == target)].iloc[0]
    return f"| {version} | {sub['n_features']} | {sub['auc_roc']:.4f} | {sub['auc_pr']:.4f} | {sub['recall_at_10']:.4f} | {sub['f1']:.4f} | {sub['elapsed_s']}s |"

# Determinar versión ganadora por target
winners = {}
for target in TARGETS:
    sub = metrics_df[metrics_df['target'] == target].sort_values('auc_roc', ascending=False)
    winners[target] = sub.iloc[0]['version']

# Diferencia max-min de AUC por target (¿significativa o no?)
for target in TARGETS:
    sub = metrics_df[metrics_df['target'] == target]
    delta = sub['auc_roc'].max() - sub['auc_roc'].min()
    winners[f'{target}_delta'] = delta

# Diferencia entre churn_14d y churn_30d
auc_14d_best = metrics_df[metrics_df['target'] == 'churn_14d']['auc_roc'].max()
auc_30d_best = metrics_df[metrics_df['target'] == 'churn_30d']['auc_roc'].max()

now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

lines = [
    '# Resumen ejecutivo — Baseline LogReg',
    '',
    f'**Fecha**: {now_str}',
    f'**Tiempo total**: {sum(r["elapsed_seconds"] for r in results.values()):.1f}s '
    f'(media {sum(r["elapsed_seconds"] for r in results.values())/8:.1f}s por modelo)',
    '',
    '## Setup',
    '',
    '- **8 modelos**: 4 versiones de master × 2 targets (churn_14d, churn_30d)',
    '- **Splits**: 70/15/15 estratificados por churn_30d, random_state=42, compartidos',
    '- **Modelo**: LogReg defaults razonables, early stopping (50 rondas, métrica AUC)',
    '- **Sin tuning de hiperparámetros** — baseline puro',
    '',
    '## Resultados — churn_30d',
    '',
    '| Versión | n_cols | AUC-ROC | AUC-PR | Recall@10% | F1 | Tiempo |',
    '|---|---|---|---|---|---|---|',
]
for v in sorted(MASTER_VERSIONS.keys(),
                key=lambda x: -metrics_df[(metrics_df['version']==x) & (metrics_df['target']=='churn_30d')]['auc_roc'].iloc[0]):
    lines.append(metric_row(v, 'churn_30d', metrics_df))

lines += [
    '',
    '## Resultados — churn_14d',
    '',
    '| Versión | n_cols | AUC-ROC | AUC-PR | Recall@10% | F1 | Tiempo |',
    '|---|---|---|---|---|---|---|',
]
for v in sorted(MASTER_VERSIONS.keys(),
                key=lambda x: -metrics_df[(metrics_df['version']==x) & (metrics_df['target']=='churn_14d')]['auc_roc'].iloc[0]):
    lines.append(metric_row(v, 'churn_14d', metrics_df))

lines += [
    '',
    '## Hallazgos clave',
    '',
    '### ¿Qué versión de master gana?',
    '',
    f'- **churn_30d**: gana `{winners["churn_30d"]}` (AUC-ROC máxima)',
    f'- **churn_14d**: gana `{winners["churn_14d"]}` (AUC-ROC máxima)',
    f'- Delta AUC max-min churn_30d: **{winners["churn_30d_delta"]:.4f}**',
    f'- Delta AUC max-min churn_14d: **{winners["churn_14d_delta"]:.4f}**',
    '',
    'Interpretación:',
    '- Si delta < 0.005: **las 4 versiones funcionan prácticamente igual** → el árbol ignora las features inútiles, defendible quedarse con v3 (más ligera).',
    '- Si delta entre 0.005 y 0.02: **diferencia leve** — depende del coste/beneficio de mantener más features.',
    '- Si delta > 0.02: **diferencia material** — investigar qué features de las eliminadas aportaban señal.',
    '',
    f'En este experimento (delta_30d={winners["churn_30d_delta"]:.4f}, delta_14d={winners["churn_14d_delta"]:.4f}):',
]
for target in TARGETS:
    delta = winners[f'{target}_delta']
    if delta < 0.005:
        verdict = 'prácticamente equivalentes — quedarse con v3 (más ligera)'
    elif delta < 0.02:
        verdict = 'diferencia leve — evaluar coste/beneficio'
    else:
        verdict = 'diferencia material — investigar features eliminadas'
    lines.append(f'- {target}: {verdict}')

lines += [
    '',
    '### ¿Qué features son consistentemente importantes?',
    '',
    f'Features que aparecen en el top 30 de >=6 de los 8 modelos (ver `robust_top_features.csv`):',
    '',
    '| Feature | Modelos en top 30 |',
    '|---|---|',
]
for f, n in robust[:15]:
    lines.append(f'| `{f}` | {n}/8 |')
if len(robust) > 15:
    lines.append(f'| ... | ({len(robust) - 15} más) |')

lines += [
    '',
    '### ¿Hay diferencia entre churn_14d y churn_30d?',
    '',
    f'- AUC máxima churn_14d: **{auc_14d_best:.4f}**',
    f'- AUC máxima churn_30d: **{auc_30d_best:.4f}**',
    f'- Diferencia: {auc_14d_best - auc_30d_best:+.4f}',
    '',
]
if auc_14d_best > auc_30d_best:
    lines.append('Interpretación: **churn_14d es más predecible** que churn_30d. Coherente con la lógica:')
    lines.append('a 14 días la decisión del jugador es más "confirmada", a 30 días hay más ruido por re-engagement tardío.')
elif auc_30d_best > auc_14d_best + 0.005:
    lines.append('Interpretación: churn_30d es más predecible. Esto es contraintuitivo — investigar si la definición')
    lines.append('del target a 30d incluye señal posterior a la fecha que el modelo capta.')
else:
    lines.append('Interpretación: rendimiento similar entre 14d y 30d. Las features capturan ambos horizontes por igual.')

lines += [
    '',
    '## Próximos pasos',
    '',
    '1. **Tuning de hiperparámetros** sobre la versión ganadora (Optuna / GridSearch).',
    '2. **Otros algoritmos** como control: XGBoost, LogisticRegression con regularización.',
    '3. **SHAP** para interpretabilidad de las features top.',
    '4. **Análisis de errores**: examinar los false negatives (jugadores que churn-aron sin que el modelo lo detecte).',
    '5. **Modelo de gustos**: experimento separado, no churn (clustering / recomendación).',
    '',
]

SUMMARY_PATH = REPORT_DIR / 'baseline_summary.md'
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f'baseline_summary.md guardado: {SUMMARY_PATH}')
log_step('REPORT', '8.1 baseline_summary.md')


baseline_summary.md guardado: /Users/jezquerro/Documents/tfg/informes/fase3_modeling/03e_logreg/baseline_comparison/baseline_summary.md
[REPORT] 19:37:13 — 8.1 baseline_summary.md


In [15]:
# [REPORT] 8.2 execution_report.md (técnico completo)
total_elapsed = time.time() - NOTEBOOK_START
t_min = int(total_elapsed // 60)
t_sec = int(total_elapsed % 60)

lines = [
    '# Execution Report — 03e_baseline_logreg',
    '',
    f'**Notebook**: notebooks/fase3_modeling/03e_baseline_logreg.ipynb',
    f'**Fecha**: {now_str}',
    f'**Tiempo de ejecución total**: {t_min} min {t_sec} s',
    f'**Modelos entrenados**: 8 (4 versiones × 2 targets)',
    '',
    '## Setup',
    '',
    f'- LogReg {sklearn.__version__}',
    f'- sklearn — train_test_split estratificado',
    f'- random_state = {RANDOM_STATE}',
    f'- Splits: train={len(idx_train)} ({TRAIN_SIZE*100:.0f}%), val={len(idx_val)} ({VAL_SIZE*100:.0f}%), test={len(idx_test)} ({TEST_SIZE*100:.0f}%)',
    f'- Estratificación por `churn_30d`',
    f'- Tasas de churn 30d: train={churn_train:.4f}, val={churn_val:.4f}, test={churn_test:.4f}',
    '',
    '## Hiperparámetros LogReg (defaults razonables)',
    '',
    '| Parámetro | Valor |',
    '|---|---|',
    '| `objective` | binary |',
    '| `metric` | auc |',
    '| `n_estimators` | 500 (con early stopping a 50 rondas) |',
    '| `learning_rate` | 0.05 |',
    '| `num_leaves` | 31 |',
    '| `max_depth` | -1 (sin límite) |',
    '| `min_child_samples` | 20 |',
    '| `random_state` | 42 |',
    '',
    '## Resultados (las 8 entradas)',
    '',
    '| version | target | n_features | best_iter | AUC-ROC | AUC-PR | F1 | Recall@10 | log_loss | t (s) |',
    '|---|---|---|---|---|---|---|---|---|---|',
]
for r in metrics_df.sort_values(['target', 'auc_roc'], ascending=[True, False]).to_dict('records'):
    lines.append(
        f'| {r["version"]} | {r["target"]} | {r["n_features"]} | {r["best_iteration"]} '
        f'| {r["auc_roc"]:.4f} | {r["auc_pr"]:.4f} | {r["f1"]:.4f} | {r["recall_at_10"]:.4f} '
        f'| {r["log_loss"]:.4f} | {r["elapsed_s"]} |'
    )

lines += [
    '',
    '## Confusion matrices (test, threshold=0.5)',
    '',
    '| version | target | TN | FP | FN | TP |',
    '|---|---|---|---|---|---|',
]
for r in metrics_df.sort_values(['target', 'version']).to_dict('records'):
    lines.append(f'| {r["version"]} | {r["target"]} | {r["tn"]} | {r["fp"]} | {r["fn"]} | {r["tp"]} |')

lines += [
    '',
    '## Decisiones tomadas',
    '',
    '1. **Splits unificados**: calculados una sola vez sobre la master original y aplicados a los 8 experimentos. Garantiza comparabilidad estricta.',
    '2. **Categóricas nativas**: dtype `category` en lugar de one-hot. LogReg las maneja con su propio splitter.',
    '3. **NaN preservados**: LogReg los enruta nativamente. NO se imputa.',
    '4. **Threshold = 0.5**: sin búsqueda de óptimo en este baseline. La búsqueda va en notebook posterior.',
    '5. **Early stopping = 50** sobre AUC en validation. Evita overfitting sin invocar tuning.',
    '6. **Métricas múltiples**: AUC-ROC (clasificación binaria), AUC-PR (mejor para clases desbalanceadas), F1, Recall@10% (operativa: si solo podemos contactar al 10% top, qué % de churners cubrimos), log-loss.',
    '',
    '## Limitaciones del baseline',
    '',
    '- **Sin tuning**: los hiperparámetros son defaults; con Optuna se puede mejorar 0.005-0.02 AUC.',
    '- **Threshold fijo**: 0.5 no maximiza F1 en clases desbalanceadas. Buscar óptimo en notebook posterior.',
    '- **Solo LogReg**: falta comparación con otros modelos (XGBoost, LR, etc.).',
    '- **Sin SHAP**: feature importance por gain es global; SHAP da explicabilidad por instancia.',
    '- **Splits fijos**: cross-validation daría mejor estimación de la incertidumbre de las métricas.',
    '',
    '## Pasos ejecutados',
    '',
]
for s in steps_log:
    lines.append(f'- {s}')

lines += [
    '',
    '## TODOs para Fase 4',
    '',
    '- [ ] Tuning de hiperparámetros con Optuna sobre la versión ganadora',
    '- [ ] Comparar con XGBoost y Logistic Regression como controles',
    '- [ ] SHAP para interpretabilidad por instancia',
    '- [ ] Cross-validation k=5 sobre el train+val para estimar incertidumbre',
    '- [ ] Análisis de errores: subgrupos donde el modelo falla más',
    '- [ ] Calibración de probabilidades (Platt scaling / isotonic)',
    '- [ ] Threshold óptimo por target según métrica de negocio',
    '',
    '## Archivos generados',
    '',
    '| Archivo | Tipo |',
    '|---|---|',
    '| `data/data_models/splits_indices_cutoff.parquet` | splits compartidos |',
    '| `data/data_models/model_<v>_<t>.txt` | 8 modelos LogReg nativos |',
    '| `data/data_models/predictions_<v>_<t>.parquet` | 8 archivos de predicciones sobre test |',
    '| `informes/fase3_modeling/03a_lightgbm/baseline_comparison/metrics_comparison.csv` | tabla comparativa |',
    '| `informes/fase3_modeling/03a_lightgbm/baseline_comparison/feature_importance_<v>_<t>.csv` | 8 feature importances |',
    '| `informes/fase3_modeling/03a_lightgbm/baseline_comparison/robust_top_features.csv` | features en top 30 de >=6 modelos |',
    '| `informes/fase3_modeling/03a_lightgbm/baseline_comparison/roc_curves.png` | 4 curvas ROC × 2 targets |',
    '| `informes/fase3_modeling/03a_lightgbm/baseline_comparison/pr_curves.png` | 4 curvas PR × 2 targets |',
    '| `informes/fase3_modeling/03a_lightgbm/baseline_comparison/confusion_matrices.png` | 8 matrices |',
    '| `informes/fase3_modeling/03a_lightgbm/baseline_comparison/top_features_heatmap.png` | rank top 30 |',
    '| `informes/fase3_modeling/03a_lightgbm/baseline_comparison/baseline_summary.md` | resumen ejecutivo |',
    '| `informes/fase3_modeling/03a_lightgbm/baseline_comparison/execution_report.md` | este informe |',
    '',
]

REPORT_PATH = REPORT_DIR / 'execution_report.md'
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f'execution_report.md guardado: {REPORT_PATH}')
log_step('REPORT', '8.2 execution_report.md')


execution_report.md guardado: /Users/jezquerro/Documents/tfg/informes/fase3_modeling/03e_logreg/baseline_comparison/execution_report.md
[REPORT] 19:37:13 — 8.2 execution_report.md


In [16]:
# [REPORT] Resumen final en consola
elapsed = time.time() - NOTEBOOK_START
print('=' * 70)
print(f'RESUMEN FINAL — Notebook 03e_baseline_logreg')
print('=' * 70)
print(f'  Tiempo total                : {int(elapsed//60)}m {int(elapsed%60)}s')
print(f'  Modelos entrenados          : 8')
print()
print(f'  AUC-ROC máxima por target:')
for target in TARGETS:
    sub = metrics_df[metrics_df['target'] == target]
    best = sub.loc[sub['auc_roc'].idxmax()]
    print(f'    {target:<12s}: {best["auc_roc"]:.4f}  ({best["version"]})')
print()
print(f'  Delta AUC max-min:')
for target in TARGETS:
    sub = metrics_df[metrics_df['target'] == target]
    delta = sub['auc_roc'].max() - sub['auc_roc'].min()
    print(f'    {target:<12s}: {delta:.4f}')
print()
print(f'  Outputs:')
print(f'    data/data_models/        (splits + 8 modelos + 8 predicciones)')
print(f'    {REPORT_DIR.relative_to(PROJECT_ROOT)}/')
print(f'       (metrics, importance, gráficos, reports)')
print('=' * 70)


RESUMEN FINAL — Notebook 03e_baseline_logreg
  Tiempo total                : 0m 27s
  Modelos entrenados          : 8

  AUC-ROC máxima por target:
    churn_14d   : 0.9993  (original)
    churn_30d   : 0.9999  (original)

  Delta AUC max-min:
    churn_14d   : 0.2102
    churn_30d   : 0.2520

  Outputs:
    data/data_models/        (splits + 8 modelos + 8 predicciones)
    informes/fase3_modeling/03e_logreg/baseline_comparison/
       (metrics, importance, gráficos, reports)


In [17]:
# [REPORT] Logging dual: HTML
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase3_modeling' / '03e_baseline_logreg.ipynb'
html_path = REPORT_DIR / '03e_baseline_logreg_run.html'
export_notebook_to_html(notebook_path, html_path)

# Sección enriquecida sobre metrics_df (resumen tabular)
enriched = build_enriched_report_section(
    df_final=metrics_df,
    raw_shape=None,
    filter_steps=[
        ('Modelos planeados', 8),
        ('Modelos entrenados', len(results)),
    ],
)
with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
    f.write('\n\n---\n\n## Outputs detallados (metrics_df)\n\n' + enriched)
print(f'HTML + enriquecimiento appendeado')


HTML generado: 03e_baseline_logreg_run.html (0.5 MB)
HTML + enriquecimiento appendeado
